# SQL — Common Table Expressions (CTEs)
**Day 1 — SQL Module**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> CTEs are a <em>readability</em> feature, not a performance feature.
They name intermediate result sets so complex multi-step queries read like sequential logic
instead of nested spaghetti. The optimizer often treats them the same as subqueries — but
your colleagues (and future you) will thank you.
</div>

### Why It Matters
At Citi, capacity queries had 4-5 logical steps: filter raw telemetry → deduplicate → daily average → 30-day trend → flag at-risk servers. Without CTEs, this is 4 levels of nested subqueries. With CTEs, each step is named and readable.

## The Syntax

```sql
WITH cte_name AS (
    SELECT ...
    FROM   ...
    WHERE  ...
),
second_cte AS (
    SELECT ...
    FROM   cte_name   -- reference the first CTE by name
    WHERE  ...
)
SELECT *
FROM second_cte;
```

**Key rules:**
- Start with `WITH`, not `SELECT`
- Separate multiple CTEs with commas — the **last one has no comma**
- Reference earlier CTEs by name in later CTEs
- The final `SELECT` is the actual query that returns results — it's not part of the CTE

In [ ]:
-- Example 1: Find employees earning above their department average
-- Step 1: Calculate average salary per department (named CTE)
-- Step 2: Join back to find employees above that average

WITH dept_averages AS (
    SELECT
        department_id,
        AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department_id
)
SELECT
    e.name,
    e.salary,
    d.avg_salary,
    ROUND(e.salary - d.avg_salary, 2) AS above_average_by
FROM employees e
JOIN dept_averages d ON e.department_id = d.department_id
WHERE e.salary > d.avg_salary
ORDER BY above_average_by DESC;

-- Without CTE (nested subquery — same result, harder to read):
SELECT e.name, e.salary,
       (SELECT AVG(salary) FROM employees e2
        WHERE e2.department_id = e.department_id) AS avg_salary
FROM employees e
WHERE e.salary > (SELECT AVG(salary) FROM employees e2
                  WHERE e2.department_id = e.department_id);

## Chained CTEs — Multi-Step Pipeline

This is where CTEs shine: expressing a data pipeline as sequential, named steps.

In [ ]:
-- Capacity planning pipeline:
-- Step 1: Deduplicate raw telemetry
-- Step 2: Compute daily averages
-- Step 3: Identify at-risk servers (peak > 80%)

WITH deduped AS (
    -- Remove any duplicate records from the same server/day
    SELECT DISTINCT
        server_id,
        collection_date,
        cpu_utilization
    FROM telemetry_raw
    WHERE collection_date >= CURRENT_DATE - INTERVAL '30 days'
),
daily_avg AS (
    -- One row per server per day
    SELECT
        server_id,
        collection_date,
        AVG(cpu_utilization) AS avg_cpu,
        MAX(cpu_utilization) AS peak_cpu
    FROM deduped
    GROUP BY server_id, collection_date
),
monthly_stats AS (
    -- Aggregate to 30-day summary per server
    SELECT
        server_id,
        AVG(avg_cpu)  AS avg_30d,
        MAX(peak_cpu) AS peak_30d,
        COUNT(*)      AS days_sampled
    FROM daily_avg
    GROUP BY server_id
)
SELECT
    server_id,
    ROUND(avg_30d, 2)  AS avg_cpu_30d,
    ROUND(peak_30d, 2) AS peak_cpu_30d,
    days_sampled,
    CASE WHEN peak_30d > 80 THEN 'AT RISK'
         WHEN avg_30d  > 65 THEN 'MONITOR'
         ELSE 'HEALTHY' END AS status
FROM monthly_stats
ORDER BY peak_30d DESC;

## Recursive CTEs — Walk a Hierarchy

Use when data has a parent-child relationship: org charts, folder trees, alert escalation chains, network topology.

In [ ]:
-- Recursive CTE: Walk an org chart from CEO down to all reports

WITH RECURSIVE org_chart AS (
    -- BASE CASE: The root node (CEO has no manager)
    SELECT
        employee_id,
        name,
        manager_id,
        1 AS org_level,
        name AS path
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    -- RECURSIVE CASE: Join employees to their manager
    SELECT
        e.employee_id,
        e.name,
        e.manager_id,
        oc.org_level + 1,
        oc.path || ' > ' || e.name   -- build the path string
    FROM employees e
    JOIN org_chart oc ON e.manager_id = oc.employee_id
    WHERE oc.org_level < 10           -- SAFETY: prevent infinite loops from cycles
)
SELECT
    LPAD(' ', (org_level - 1) * 4, ' ') || name AS org_tree,
    org_level,
    path
FROM org_chart
ORDER BY path;

-- Without the depth limit, a cycle (A manages B, B manages A) would loop forever.
-- Always add WHERE level < N or a MAXRECURSION hint.

## Interview Q&A

**Q: What is the difference between a CTE and a subquery?**
A: CTEs are named and reusable within the same query — you can reference a CTE multiple times. Subqueries are inline and can't be referenced elsewhere. CTEs are significantly more readable for multi-step logic. Functionally, most optimizers treat them the same (no materialization guarantee).

**Q: Do CTEs improve performance?**
A: Not inherently. In PostgreSQL ≥ 12, the optimizer can "inline" CTEs and optimize them as if they were subqueries. In PostgreSQL < 12, CTEs were optimization fences — always materialized. In SQL Server, you can use `WITH (NOEXPAND)`. Always check your engine.

**Q: CTE vs temp table — when does temp table win?**
A: When you need to reference the result many times (e.g., in multiple subsequent queries, not just once), or when you need to index the intermediate result. Temp tables materialize and can be indexed. CTEs may or may not materialize depending on the optimizer.

**Q: When would you use a recursive CTE?**
A: Hierarchy traversal — org charts, file system paths, category trees, network hops, alert escalation chains. At Citi: "Show me the full escalation chain from a base alert up to the management team" — that's a recursive CTE on an alert_hierarchy table.

**Q: What is the risk of recursive CTEs?**
A: Infinite loops from cycles in the data. Always add a depth limit: `WHERE level < 10` or `MAXRECURSION 100` (SQL Server). In production, also add cycle detection if the data might have genuine cycles.

## The Citi Angle

**The capacity pipeline above is a real pattern.** At Citi, the multi-step telemetry query was:
1. Filter last 90 days of raw APM data
2. Deduplicate (APM agents sometimes sent duplicate records)
3. Daily aggregates (avg, max, percentile)
4. Trend calculation (linear regression or 14-day moving average)
5. Flag at-risk: projected to exceed capacity in < 35 days

Without CTEs: a query with 4 nested subqueries, unreadable and untestable.
With CTEs: each step named, each step independently testable.

**Interview line:** *"At Citi, I wrote a 5-step CTE chain that processed 500M telemetry rows to identify capacity risks. The CTE structure meant each pipeline stage was readable and independently debuggable — critical when troubleshooting data quality issues in production."*

```sql
-- The debugging superpower: test any step in isolation
WITH deduped AS ( ... ),
daily_avg AS ( SELECT ... FROM deduped ... )
SELECT * FROM daily_avg LIMIT 100;  -- examine intermediate step
```